# Feature Engineering - DDI Risk Analysis
This notebook creates features for patient-level DDI risk analysis and clustering.

**Approach:** Dual feature sets  
- **Patient-level features**: For clustering and risk scoring  
- **DDI pair-level features**: For detailed interaction analysis

**Input**:  
- `med-data/v2_clean/ddi/db_drug_interactions_clean.parquet`  
- `med-data/v2_clean/medications/medications_clean.parquet`

**Output**:  
- `med-data/v3_features/patients_features.parquet` (one row per patient)  
- `med-data/v3_features/ddi_pairs_features.parquet` (one row per patient DDI pair)

**Future Enhancement**: After adding PhysioNet MIMIC-IV data, analyze care coordination  
risks between VA and non-VA settings (fragmented care DDI analysis).

In [2]:
# Import dependencies

import os
import sys
import logging
import time
import re
from datetime import datetime, timedelta
from itertools import combinations
import numpy as np
import pandas as pd
import s3fs
import pyarrow as pa
from scipy.stats import entropy
from importlib.metadata import version
from config import *

In [3]:
# Verify dependencies

def print_version():
    print("pandas:", pd.__version__)
    print("numpy:", np.__version__)
    print("scipy:", version("scipy"))
    print("s3fs:", s3fs.__version__)
    print("pyarrow:", pa.__version__)

print_version()

pandas: 2.3.3
numpy: 2.3.4
scipy: 1.16.3
s3fs: 2025.10.0
pyarrow: 22.0.0


In [4]:
# Set up logging

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)

logging.info("Logging configured successfully")

2025-11-27 12:48:05,766 INFO Logging configured successfully


In [5]:
# Load configuration

logging.info(f"MinIO endpoint: {MINIO_ENDPOINT}")
logging.info(f"Source: {DEST_BUCKET}/v2_clean/")
logging.info(f"Destination: {DEST_BUCKET}/v3_features/")

2025-11-27 12:48:09,593 INFO MinIO endpoint: localhost:9000
2025-11-27 12:48:09,594 INFO Source: med-data/v2_clean/
2025-11-27 12:48:09,594 INFO Destination: med-data/v3_features/


In [6]:
# Create S3FileSystem for MinIO

logging.info(f"Initializing S3FileSystem for MinIO at {MINIO_ENDPOINT}")
fs = s3fs.S3FileSystem(
    anon=False,
    key=MINIO_ACCESS_KEY,
    secret=MINIO_SECRET_KEY,
    client_kwargs={'endpoint_url': f"http://{MINIO_ENDPOINT}"}
)
logging.info("S3FileSystem created successfully")

2025-11-27 12:48:11,444 INFO Initializing S3FileSystem for MinIO at localhost:9000
2025-11-27 12:48:11,446 INFO S3FileSystem created successfully


---
## Part 1: Load Clean Data

In [7]:
# Load DDI clean dataset from v2_clean

ddi_uri = f"s3://{DEST_BUCKET}/{V2_CLEAN_DDI_PREFIX}db_drug_interactions_clean.parquet"
logging.info(f"Reading DDI clean data: {ddi_uri}")

start_time = time.time()
df_ddi = pd.read_parquet(ddi_uri, filesystem=fs)
elapsed = time.time() - start_time

logging.info(f"Loaded {len(df_ddi):,} DDI records in {elapsed:.2f}s")

print(f"DDI Data Shape: {df_ddi.shape}")
print(f"Columns: {list(df_ddi.columns)}")
df_ddi.head(3)

2025-11-27 12:48:14,655 INFO Reading DDI clean data: s3://med-data/v2_clean/ddi/db_drug_interactions_clean.parquet
2025-11-27 12:48:14,833 INFO Loaded 191,541 DDI records in 0.18s


DDI Data Shape: (191541, 6)
Columns: ['Drug 1', 'Drug 2', 'Interaction Description', 'Drug1_Normalized', 'Drug2_Normalized', 'Severity']


,Drug 1,Drug 2,Interaction Description,Drug1_Normalized,Drug2_Normalized,Severity
0,Trioxsalen,Verteporfin,Trioxsalen may increase the photosensitizing activities of Verteporfin.,TRIOXSALEN,VERTEPORFIN,Moderate
1,Aminolevulinic acid,Verteporfin,Aminolevulinic acid may increase the photosensitizing activities of Verteporfin.,AMINOLEVULINIC ACID,VERTEPORFIN,Moderate
2,Titanium dioxide,Verteporfin,Titanium dioxide may increase the photosensitizing activities of Verteporfin.,TITANIUM DIOXIDE,VERTEPORFIN,Moderate


In [8]:
# Load medications clean dataset from v2_clean

meds_uri = f"s3://{DEST_BUCKET}/{V2_CLEAN_MEDICATIONS_PREFIX}medications_clean.parquet"
logging.info(f"Reading medications clean data: {meds_uri}")

start_time = time.time()
df_meds = pd.read_parquet(meds_uri, filesystem=fs)
elapsed = time.time() - start_time

logging.info(f"Loaded {len(df_meds):,} medication records in {elapsed:.2f}s")

print(f"\nMedications Data Shape: {df_meds.shape}")
print(f"Columns: {list(df_meds.columns)}")
df_meds.head(3)

2025-11-27 12:48:20,109 INFO Reading medications clean data: s3://med-data/v2_clean/medications/medications_clean.parquet
2025-11-27 12:48:20,141 INFO Loaded 36 medication records in 0.03s



Medications Data Shape: (36, 23)
Columns: ['PatientSID', 'PatientIEN', 'Sta3n', 'DrugNameWithoutDose', 'DrugNameWithDose', 'SourceSystem', 'MedicationDateTime', 'StartDate', 'EndDate', 'Status', 'DaysSupply', 'Quantity', 'DEASchedule', 'ControlledSubstanceFlag', 'OrderNumber', 'ProviderSID', 'LocalDrugSID', 'NationalDrugSID', 'DrugName_Normalized', 'MedicationYear', 'MedicationMonth', 'MedicationDayOfWeek', 'MedicationHour']


,PatientSID,PatientIEN,Sta3n,DrugNameWithoutDose,DrugNameWithDose,SourceSystem,MedicationDateTime,StartDate,EndDate,Status,...,ControlledSubstanceFlag,OrderNumber,ProviderSID,LocalDrugSID,NationalDrugSID,DrugName_Normalized,MedicationYear,MedicationMonth,MedicationDayOfWeek,MedicationHour
0,1001,PtIEN1001,508,LISINOPRIL,LISINOPRIL 10MG TAB,BCMA,2025-01-01 08:05:00,2025-01-01 08:05:00,NaT,GIVEN,...,None,IP-2025-001001,1001,10002,20002,LISINOPRIL,2025,1,2,8
1,1001,PtIEN1001,508,METFORMIN HCL,METFORMIN HCL 500MG TAB,BCMA,2025-01-01 12:10:00,2025-01-01 12:10:00,NaT,GIVEN,...,None,IP-2025-001002,1001,10001,20001,METFORMIN,2025,1,2,12
2,1001,PtIEN1001,508,METFORMIN HCL,METFORMIN HCL 500MG TAB,BCMA,2025-01-01 18:08:00,2025-01-01 18:08:00,NaT,GIVEN,...,None,IP-2025-001002,1001,10001,20001,METFORMIN,2025,1,2,18


---
## Part 2: Patient-Level Features

Create aggregated features for each patient to support clustering and risk scoring.

In [27]:
# Calculate medication profile features per patient

logging.info("Calculating patient medication profile features...")

# Group by patient
patient_features = df_meds.groupby('PatientSID').agg(
    medication_count=('DrugName_Normalized', 'count'),
    unique_medications=('DrugName_Normalized', 'nunique'),
    first_medication_date=('MedicationDateTime', 'min'),
    last_medication_date=('MedicationDateTime', 'max'),
    rxout_count=('SourceSystem', lambda x: (x == 'RxOut').sum()),
    bcma_count=('SourceSystem', lambda x: (x == 'BCMA').sum())
).reset_index()

# Calculate medication timespan
patient_features['medication_timespan_days'] = (
    patient_features['last_medication_date'] - patient_features['first_medication_date']
).dt.days

# Calculate average medications per day (medication burden)
patient_features['avg_medications_per_day'] = (
    patient_features['medication_count'] / 
    (patient_features['medication_timespan_days'] + 1)  # +1 to avoid division by zero
)

# Calculate source system diversity (0 = single source, higher = more diverse)
patient_features['source_diversity'] = (
    (patient_features['rxout_count'] > 0).astype(int) + 
    (patient_features['bcma_count'] > 0).astype(int)
)

logging.info(f"Created medication profile features for {len(patient_features)} patients")

print("\nPatient Medication Profile Features:")
patient_features.head()

2025-11-27 12:58:14,070 INFO Calculating patient medication profile features...
2025-11-27 12:58:14,076 INFO Created medication profile features for 10 patients



Patient Medication Profile Features:


,PatientSID,medication_count,unique_medications,first_medication_date,last_medication_date,rxout_count,bcma_count,medication_timespan_days,avg_medications_per_day,source_diversity
0,1001,7,3,2025-01-01 08:05:00,2025-03-01 10:00:00,3,4,59,0.116667,2
1,1002,3,3,2025-01-02 07:30:00,2025-01-22 09:15:00,1,2,20,0.142857,2
2,1003,3,2,2025-01-02 21:10:00,2025-02-05 13:25:00,2,1,33,0.088235,2
3,1004,4,3,2025-01-01 14:15:00,2025-03-12 15:30:00,1,3,70,0.056338,2
4,1005,4,3,2025-01-03 17:05:00,2025-02-01 10:00:00,3,1,28,0.137931,2


In [28]:
# Calculate medication diversity using Shannon entropy

logging.info("Calculating medication diversity scores...")

def calculate_medication_diversity(patient_meds):
    """Calculate Shannon entropy of medication distribution for a patient."""
    med_counts = patient_meds['DrugName_Normalized'].value_counts()
    if len(med_counts) <= 1:
        return 0.0
    return entropy(med_counts, base=2)

diversity_scores = df_meds.groupby('PatientSID').apply(calculate_medication_diversity)
diversity_scores = diversity_scores.reset_index(name='medication_diversity')

# Merge with patient features
patient_features = patient_features.merge(diversity_scores, on='PatientSID', how='left')

logging.info("Medication diversity scores added")

print("\nMedication diversity distribution:")
print(patient_features['medication_diversity'].describe())

2025-11-27 12:58:20,789 INFO Calculating medication diversity scores...
/var/folders/n3/9zcf9kqj3rsb59rwpwsmy2jh0000gn/T/ipykernel_30887/1059576220.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  diversity_scores = df_meds.groupby('PatientSID').apply(calculate_medication_diversity)
2025-11-27 12:58:20,797 INFO Medication diversity scores added



Medication diversity distribution:
count    10.000000
mean      1.187037
std       0.280305
min       0.918296
25%       1.000000
50%       1.000000
75%       1.487204
max       1.584963
Name: medication_diversity, dtype: float64


In [29]:
# Identify all DDI pairs for each patient

logging.info("Identifying DDI pairs for each patient...")

def find_patient_ddi_pairs(patient_id, patient_meds_df, ddi_df):
    """
    Find all DDI pairs for a given patient.
    Returns list of dicts with interaction details.
    """
    # Get patient's unique medications
    meds = patient_meds_df[patient_meds_df['PatientSID'] == patient_id]['DrugName_Normalized'].dropna().unique()
    
    interactions = []
    
    # Check all pairs of patient's medications
    for drug1, drug2 in combinations(meds, 2):
        # Check if this pair exists in DDI dataset (either order)
        match = ddi_df[
            ((ddi_df['Drug1_Normalized'] == drug1) & (ddi_df['Drug2_Normalized'] == drug2)) |
            ((ddi_df['Drug1_Normalized'] == drug2) & (ddi_df['Drug2_Normalized'] == drug1))
        ]
        
        if not match.empty:
            for _, row in match.iterrows():
                interactions.append({
                    'PatientSID': patient_id,
                    'Drug1': drug1,
                    'Drug2': drug2,
                    'Severity': row['Severity'],
                    'Interaction': row['Interaction Description']
                })
    
    return interactions

# Find DDI pairs for all patients
all_patient_ddis = []
for patient_id in df_meds['PatientSID'].unique():
    patient_ddis = find_patient_ddi_pairs(patient_id, df_meds, df_ddi)
    all_patient_ddis.extend(patient_ddis)

# Create DataFrame of patient DDI pairs
df_patient_ddis = pd.DataFrame(all_patient_ddis)

logging.info(f"Found {len(df_patient_ddis)} total DDI pairs across all patients")

if len(df_patient_ddis) > 0:
    print(f"\nDDI pairs found: {len(df_patient_ddis)}")
    print(f"Patients with DDIs: {df_patient_ddis['PatientSID'].nunique()}")
    print("\nSeverity distribution:")
    print(df_patient_ddis['Severity'].value_counts())
    print("\nSample DDI pairs:")
    print(df_patient_ddis.head())
else:
    print("\n⚠ No DDI pairs found in current patient data")

2025-11-27 12:58:26,170 INFO Identifying DDI pairs for each patient...
2025-11-27 12:58:26,434 INFO Found 4 total DDI pairs across all patients



DDI pairs found: 4
Patients with DDIs: 3

Severity distribution:
Severity
Low         2
Moderate    2
Name: count, dtype: int64

Sample DDI pairs:
   PatientSID       Drug1                 Drug2  Severity  \
0        1001  LISINOPRIL        SPIRONOLACTONE       Low   
1        1003  SERTRALINE            ALPRAZOLAM       Low   
2        1005    WARFARIN  ACETYLSALICYLIC ACID  Moderate   
3        1005  METOPROLOL  ACETYLSALICYLIC ACID  Moderate   

                                                                                                 Interaction  
0  The risk or severity of adverse effects can be increased when Spironolactone is combined with Lisinopril.  
1      The risk or severity of adverse effects can be increased when Alprazolam is combined with Sertraline.  
2                                Warfarin may increase the anticoagulant activities of Acetylsalicylic acid.  
3                           Metoprolol may decrease the antihypertensive activities of Acetylsalicylic

In [30]:
# Calculate DDI risk features per patient

logging.info("Calculating patient DDI risk features...")

if len(df_patient_ddis) > 0:
    # Count DDI pairs by severity for each patient
    ddi_counts = df_patient_ddis.groupby(['PatientSID', 'Severity']).size().unstack(fill_value=0)
    ddi_counts = ddi_counts.add_prefix('ddi_severity_').reset_index()
    
    # Total DDI pair count
    ddi_total = df_patient_ddis.groupby('PatientSID').size().reset_index(name='ddi_pair_count')
    
    # Merge DDI features
    ddi_features = ddi_total.merge(ddi_counts, on='PatientSID', how='left')
    
    # Calculate weighted DDI risk score (High=3, Moderate=2, Low=1)
    severity_weights = {'High': 3, 'Moderate': 2, 'Low': 1}
    ddi_features['total_ddi_risk_score'] = 0
    for severity, weight in severity_weights.items():
        col_name = f'ddi_severity_{severity}'
        if col_name in ddi_features.columns:
            ddi_features['total_ddi_risk_score'] += ddi_features[col_name] * weight
    
    # Maximum severity level (3=High, 2=Moderate, 1=Low, 0=None)
    def get_max_severity(row):
        if 'ddi_severity_High' in row and row.get('ddi_severity_High', 0) > 0:
            return 3
        elif 'ddi_severity_Moderate' in row and row.get('ddi_severity_Moderate', 0) > 0:
            return 2
        elif 'ddi_severity_Low' in row and row.get('ddi_severity_Low', 0) > 0:
            return 1
        return 0
    
    ddi_features['max_severity_level'] = ddi_features.apply(get_max_severity, axis=1)
    
    # Merge with patient features
    patient_features = patient_features.merge(ddi_features, on='PatientSID', how='left')
    
    logging.info(f"Added DDI risk features for {len(ddi_features)} patients with DDIs")
else:
    # Add empty DDI columns if no DDIs found
    patient_features['ddi_pair_count'] = 0
    patient_features['total_ddi_risk_score'] = 0
    patient_features['max_severity_level'] = 0
    logging.warning("No DDI pairs found - added zero-value DDI risk features")

# Fill NaN values with 0 for patients without DDIs
ddi_cols = [col for col in patient_features.columns if 'ddi' in col.lower()]
patient_features[ddi_cols] = patient_features[ddi_cols].fillna(0)

print("\nPatient DDI Risk Features:")
print(patient_features[['PatientSID', 'unique_medications', 'ddi_pair_count', 
                         'total_ddi_risk_score', 'max_severity_level']].head())

2025-11-27 12:58:32,099 INFO Calculating patient DDI risk features...
2025-11-27 12:58:32,105 INFO Added DDI risk features for 3 patients with DDIs



Patient DDI Risk Features:
   PatientSID  unique_medications  ddi_pair_count  total_ddi_risk_score  \
0        1001                   3             1.0                   1.0   
1        1002                   3             0.0                   0.0   
2        1003                   2             1.0                   1.0   
3        1004                   3             0.0                   0.0   
4        1005                   3             2.0                   4.0   

   max_severity_level  
0                 1.0  
1                 NaN  
2                 1.0  
3                 NaN  
4                 2.0  


In [31]:
# Calculate DDI density (proportion of possible pairs that are DDIs)

logging.info("Calculating DDI density scores...")

def calculate_ddi_density(row):
    """Calculate DDI density: actual DDI pairs / total possible pairs."""
    n_meds = row['unique_medications']
    if n_meds < 2:
        return 0.0
    
    # Total possible pairs: n choose 2 = n*(n-1)/2
    total_possible_pairs = (n_meds * (n_meds - 1)) / 2
    
    ddi_pairs = row['ddi_pair_count']
    
    return ddi_pairs / total_possible_pairs if total_possible_pairs > 0 else 0.0

patient_features['ddi_density'] = patient_features.apply(calculate_ddi_density, axis=1)

logging.info("DDI density scores calculated")

print("\nDDI density distribution:")
print(patient_features['ddi_density'].describe())

2025-11-27 12:58:36,401 INFO Calculating DDI density scores...
2025-11-27 12:58:36,402 INFO DDI density scores calculated



DDI density distribution:
count    10.000000
mean      0.200000
std       0.358323
min       0.000000
25%       0.000000
50%       0.000000
75%       0.250000
max       1.000000
Name: ddi_density, dtype: float64


In [32]:
# Add polypharmacy indicator (commonly defined as 5+ medications)

logging.info("Adding polypharmacy indicators...")

patient_features['is_polypharmacy'] = (patient_features['unique_medications'] >= 5).astype(int)
patient_features['is_high_ddi_risk'] = (patient_features['max_severity_level'] >= 2).astype(int)  # Moderate or High

polypharmacy_count = patient_features['is_polypharmacy'].sum()
high_risk_count = patient_features['is_high_ddi_risk'].sum()

logging.info(f"Polypharmacy patients: {polypharmacy_count}")
logging.info(f"High DDI risk patients: {high_risk_count}")

print(f"\nPolypharmacy patients (5+ medications): {polypharmacy_count} / {len(patient_features)}")
print(f"High DDI risk patients (moderate/high severity): {high_risk_count} / {len(patient_features)}")

2025-11-27 12:58:39,079 INFO Adding polypharmacy indicators...
2025-11-27 12:58:39,081 INFO Polypharmacy patients: 0
2025-11-27 12:58:39,082 INFO High DDI risk patients: 1



Polypharmacy patients (5+ medications): 0 / 10
High DDI risk patients (moderate/high severity): 1 / 10


In [33]:
# Patient-level features summary

print("\n" + "="*80)
print("PATIENT-LEVEL FEATURES SUMMARY")
print("="*80)

print(f"\nTotal patients: {len(patient_features)}")
print(f"Total features: {len(patient_features.columns)}")

print("\nFeature groups:")
print("  - Medication profile: medication_count, unique_medications, medication_diversity")
print("  - Temporal: first/last_medication_date, medication_timespan_days, avg_medications_per_day")
print("  - Source system: rxout_count, bcma_count, source_diversity")
print("  - DDI risk: ddi_pair_count, severity counts, total_ddi_risk_score, max_severity_level")
print("  - DDI metrics: ddi_density")
print("  - Indicators: is_polypharmacy, is_high_ddi_risk")

print("\nFeature statistics:")
print(patient_features[['unique_medications', 'ddi_pair_count', 'total_ddi_risk_score', 
                         'ddi_density', 'medication_diversity']].describe())

print("="*80)

patient_features.head()


PATIENT-LEVEL FEATURES SUMMARY

Total patients: 10
Total features: 19

Feature groups:
  - Medication profile: medication_count, unique_medications, medication_diversity
  - Temporal: first/last_medication_date, medication_timespan_days, avg_medications_per_day
  - Source system: rxout_count, bcma_count, source_diversity
  - DDI risk: ddi_pair_count, severity counts, total_ddi_risk_score, max_severity_level
  - DDI metrics: ddi_density
  - Indicators: is_polypharmacy, is_high_ddi_risk

Feature statistics:
       unique_medications  ddi_pair_count  total_ddi_risk_score  ddi_density  \
count           10.000000       10.000000             10.000000    10.000000   
mean             2.400000        0.400000              0.600000     0.200000   
std              0.516398        0.699206              1.264911     0.358323   
min              2.000000        0.000000              0.000000     0.000000   
25%              2.000000        0.000000              0.000000     0.000000   
50%     

,PatientSID,medication_count,unique_medications,first_medication_date,last_medication_date,rxout_count,bcma_count,medication_timespan_days,avg_medications_per_day,source_diversity,medication_diversity,ddi_pair_count,ddi_severity_Low,ddi_severity_Moderate,total_ddi_risk_score,max_severity_level,ddi_density,is_polypharmacy,is_high_ddi_risk
0,1001,7,3,2025-01-01 08:05:00,2025-03-01 10:00:00,3,4,59,0.116667,2,1.448816,1.0,1.0,0.0,1.0,1.0,0.333333,0,0
1,1002,3,3,2025-01-02 07:30:00,2025-01-22 09:15:00,1,2,20,0.142857,2,1.584963,0.0,0.0,0.0,0.0,NaN,0.000000,0,0
2,1003,3,2,2025-01-02 21:10:00,2025-02-05 13:25:00,2,1,33,0.088235,2,0.918296,1.0,1.0,0.0,1.0,1.0,1.000000,0,0
3,1004,4,3,2025-01-01 14:15:00,2025-03-12 15:30:00,1,3,70,0.056338,2,1.500000,0.0,0.0,0.0,0.0,NaN,0.000000,0,0
4,1005,4,3,2025-01-03 17:05:00,2025-02-01 10:00:00,3,1,28,0.137931,2,1.500000,2.0,0.0,2.0,4.0,2.0,0.666667,0,1


---
## Part 3: DDI Pair-Level Features

Create detailed features for each patient-specific DDI pair.

In [34]:
# Create DDI pair-level feature dataset

logging.info("Creating DDI pair-level features...")

if len(df_patient_ddis) > 0:
    # Start with patient DDI pairs
    df_ddi_pairs = df_patient_ddis.copy()
    
    # Add patient context features
    patient_context = patient_features[['PatientSID', 'unique_medications', 'medication_count', 
                                         'total_ddi_risk_score', 'is_polypharmacy']]
    df_ddi_pairs = df_ddi_pairs.merge(patient_context, on='PatientSID', how='left')
    
    # Rename for clarity
    df_ddi_pairs = df_ddi_pairs.rename(columns={
        'unique_medications': 'patient_medication_count',
        'medication_count': 'patient_total_records',
        'total_ddi_risk_score': 'patient_total_risk_score',
        'is_polypharmacy': 'patient_is_polypharmacy'
    })
    
    logging.info(f"Created DDI pair features for {len(df_ddi_pairs)} interactions")
    
    print(f"\nDDI Pair Features Shape: {df_ddi_pairs.shape}")
    print("\nSample DDI pair features:")
    print(df_ddi_pairs.head())
else:
    df_ddi_pairs = pd.DataFrame()  # Empty dataframe
    logging.warning("No DDI pairs found - creating empty DDI pair features dataset")
    print("\n⚠ No DDI pairs to create features for")

2025-11-27 12:58:49,001 INFO Creating DDI pair-level features...
2025-11-27 12:58:49,005 INFO Created DDI pair features for 4 interactions



DDI Pair Features Shape: (4, 9)

Sample DDI pair features:
   PatientSID       Drug1                 Drug2  Severity  \
0        1001  LISINOPRIL        SPIRONOLACTONE       Low   
1        1003  SERTRALINE            ALPRAZOLAM       Low   
2        1005    WARFARIN  ACETYLSALICYLIC ACID  Moderate   
3        1005  METOPROLOL  ACETYLSALICYLIC ACID  Moderate   

                                                                                                 Interaction  \
0  The risk or severity of adverse effects can be increased when Spironolactone is combined with Lisinopril.   
1      The risk or severity of adverse effects can be increased when Alprazolam is combined with Sertraline.   
2                                Warfarin may increase the anticoagulant activities of Acetylsalicylic acid.   
3                           Metoprolol may decrease the antihypertensive activities of Acetylsalicylic acid.   

   patient_medication_count  patient_total_records  patient_total_risk_sc

In [35]:
# Calculate temporal overlap for DDI pairs

if len(df_ddi_pairs) > 0:
    logging.info("Calculating temporal overlap for DDI pairs...")
    
    def calculate_temporal_overlap(row, meds_df):
        """Calculate temporal overlap between two drugs for a patient."""
        patient_id = row['PatientSID']
        drug1 = row['Drug1']
        drug2 = row['Drug2']
        
        # Get medication records for each drug
        drug1_records = meds_df[
            (meds_df['PatientSID'] == patient_id) & 
            (meds_df['DrugName_Normalized'] == drug1)
        ]
        drug2_records = meds_df[
            (meds_df['PatientSID'] == patient_id) & 
            (meds_df['DrugName_Normalized'] == drug2)
        ]
        
        if len(drug1_records) == 0 or len(drug2_records) == 0:
            return pd.Series({
                'temporal_overlap': 0,
                'first_occurrence_date': None,
                'drug1_first_date': None,
                'drug2_first_date': None
            })
        
        # Get date ranges
        drug1_first = drug1_records['MedicationDateTime'].min()
        drug1_last = drug1_records['MedicationDateTime'].max()
        drug2_first = drug2_records['MedicationDateTime'].min()
        drug2_last = drug2_records['MedicationDateTime'].max()
        
        # Calculate overlap
        overlap_start = max(drug1_first, drug2_first)
        overlap_end = min(drug1_last, drug2_last)
        
        has_overlap = 1 if overlap_start <= overlap_end else 0
        
        # First occurrence is when both drugs are active
        first_occurrence = overlap_start if has_overlap else max(drug1_first, drug2_first)
        
        return pd.Series({
            'temporal_overlap': has_overlap,
            'first_occurrence_date': first_occurrence,
            'drug1_first_date': drug1_first,
            'drug2_first_date': drug2_first
        })
    
    # Apply temporal overlap calculation
    temporal_features = df_ddi_pairs.apply(lambda row: calculate_temporal_overlap(row, df_meds), axis=1)
    df_ddi_pairs = pd.concat([df_ddi_pairs, temporal_features], axis=1)
    
    # Calculate days between drug starts
    df_ddi_pairs['days_between_drug_starts'] = (
        df_ddi_pairs['drug2_first_date'] - df_ddi_pairs['drug1_first_date']
    ).dt.days.abs()
    
    overlap_count = df_ddi_pairs['temporal_overlap'].sum()
    logging.info(f"DDI pairs with temporal overlap: {overlap_count} / {len(df_ddi_pairs)}")
    
    print(f"\nTemporal overlap: {overlap_count} / {len(df_ddi_pairs)} DDI pairs have concurrent use")
    print("\nDays between drug starts:")
    print(df_ddi_pairs['days_between_drug_starts'].describe())

2025-11-27 12:58:52,703 INFO Calculating temporal overlap for DDI pairs...
2025-11-27 12:58:52,710 INFO DDI pairs with temporal overlap: 0 / 4



Temporal overlap: 0 / 4 DDI pairs have concurrent use

Days between drug starts:
count     4.000000
mean     31.750000
std      21.375609
min       7.000000
25%      22.750000
50%      30.500000
75%      39.500000
max      59.000000
Name: days_between_drug_starts, dtype: float64


In [36]:
# Extract interaction type from description

if len(df_ddi_pairs) > 0:
    logging.info("Extracting interaction types...")
    
    def extract_interaction_type(description):
        """Extract primary interaction mechanism from description."""
        if pd.isna(description):
            return 'Unknown'
        
        desc_lower = description.lower()
        
        # Check for common interaction types
        if 'bleeding' in desc_lower or 'anticoagulant' in desc_lower:
            return 'Bleeding Risk'
        elif 'hyperkalemia' in desc_lower or 'potassium' in desc_lower:
            return 'Hyperkalemia'
        elif 'serotonin' in desc_lower:
            return 'Serotonin Syndrome'
        elif 'nephrotoxic' in desc_lower or 'kidney' in desc_lower:
            return 'Nephrotoxicity'
        elif 'hepatotoxic' in desc_lower or 'liver' in desc_lower:
            return 'Hepatotoxicity'
        elif 'qtc' in desc_lower or 'qt prolong' in desc_lower:
            return 'QT Prolongation'
        elif 'serum concentration' in desc_lower:
            return 'Altered Drug Levels'
        elif 'adverse effect' in desc_lower:
            return 'Additive Adverse Effects'
        else:
            return 'Other'
    
    df_ddi_pairs['interaction_type'] = df_ddi_pairs['Interaction'].apply(extract_interaction_type)
    
    logging.info("Interaction types extracted")
    
    print("\nInteraction type distribution:")
    print(df_ddi_pairs['interaction_type'].value_counts())

2025-11-27 12:58:57,401 INFO Extracting interaction types...
2025-11-27 12:58:57,403 INFO Interaction types extracted



Interaction type distribution:
interaction_type
Additive Adverse Effects    2
Bleeding Risk               1
Other                       1
Name: count, dtype: int64


In [37]:
# DDI pair-level features summary

print("\n" + "="*80)
print("DDI PAIR-LEVEL FEATURES SUMMARY")
print("="*80)

if len(df_ddi_pairs) > 0:
    print(f"\nTotal DDI pairs: {len(df_ddi_pairs)}")
    print(f"Total features: {len(df_ddi_pairs.columns)}")
    print(f"Patients with DDIs: {df_ddi_pairs['PatientSID'].nunique()}")
    
    print("\nFeature groups:")
    print("  - Identification: PatientSID, Drug1, Drug2")
    print("  - Interaction: Severity, interaction_type, Interaction (description)")
    print("  - Temporal: temporal_overlap, first_occurrence_date, days_between_drug_starts")
    print("  - Patient context: patient_medication_count, patient_total_risk_score, patient_is_polypharmacy")
    
    print("\nSeverity distribution:")
    print(df_ddi_pairs['Severity'].value_counts())
    
    print("\nColumns:")
    print(list(df_ddi_pairs.columns))
else:
    print("\n⚠ No DDI pairs in dataset")

print("="*80)


DDI PAIR-LEVEL FEATURES SUMMARY

Total DDI pairs: 4
Total features: 15
Patients with DDIs: 3

Feature groups:
  - Identification: PatientSID, Drug1, Drug2
  - Interaction: Severity, interaction_type, Interaction (description)
  - Temporal: temporal_overlap, first_occurrence_date, days_between_drug_starts
  - Patient context: patient_medication_count, patient_total_risk_score, patient_is_polypharmacy

Severity distribution:
Severity
Low         2
Moderate    2
Name: count, dtype: int64

Columns:
['PatientSID', 'Drug1', 'Drug2', 'Severity', 'Interaction', 'patient_medication_count', 'patient_total_records', 'patient_total_risk_score', 'patient_is_polypharmacy', 'temporal_overlap', 'first_occurrence_date', 'drug1_first_date', 'drug2_first_date', 'days_between_drug_starts', 'interaction_type']


---
## Part 4: Feature Validation

In [38]:
# Validate patient-level features

print("="*80)
print("PATIENT-LEVEL FEATURES VALIDATION")
print("="*80)

# Check for missing values
print("\nMissing values:")
missing = patient_features.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "None")

# Check for infinite values
numeric_cols = patient_features.select_dtypes(include=[np.number]).columns
inf_check = patient_features[numeric_cols].isin([np.inf, -np.inf]).sum()
print("\nInfinite values:")
print(inf_check[inf_check > 0] if inf_check.sum() > 0 else "None")

# Check for negative values where they shouldn't be
count_cols = [col for col in patient_features.columns if 'count' in col.lower()]
negative_counts = (patient_features[count_cols] < 0).sum()
print("\nNegative count values:")
print(negative_counts[negative_counts > 0] if negative_counts.sum() > 0 else "None")

# Distribution summary
print("\nKey feature distributions:")
print(patient_features[['unique_medications', 'ddi_pair_count', 'total_ddi_risk_score']].describe())

print("="*80)

PATIENT-LEVEL FEATURES VALIDATION

Missing values:
max_severity_level    7
dtype: int64

Infinite values:
None

Negative count values:
None

Key feature distributions:
       unique_medications  ddi_pair_count  total_ddi_risk_score
count           10.000000       10.000000             10.000000
mean             2.400000        0.400000              0.600000
std              0.516398        0.699206              1.264911
min              2.000000        0.000000              0.000000
25%              2.000000        0.000000              0.000000
50%              2.000000        0.000000              0.000000
75%              3.000000        0.750000              0.750000
max              3.000000        2.000000              4.000000


In [39]:
# Validate DDI pair-level features

if len(df_ddi_pairs) > 0:
    print("="*80)
    print("DDI PAIR-LEVEL FEATURES VALIDATION")
    print("="*80)
    
    # Check for missing values
    print("\nMissing values:")
    missing = df_ddi_pairs.isnull().sum()
    print(missing[missing > 0] if missing.sum() > 0 else "None")
    
    # Check severity values are valid
    valid_severities = ['High', 'Moderate', 'Low', 'Unknown']
    invalid_severity = ~df_ddi_pairs['Severity'].isin(valid_severities)
    print(f"\nInvalid severity values: {invalid_severity.sum()}")
    
    # Check temporal overlap consistency
    print(f"\nTemporal overlap: {df_ddi_pairs['temporal_overlap'].sum()} / {len(df_ddi_pairs)} pairs")
    
    print("="*80)

DDI PAIR-LEVEL FEATURES VALIDATION

Missing values:
None

Invalid severity values: 0

Temporal overlap: 0 / 4 pairs


In [40]:
# Check correlations between key patient features

print("\nCorrelation analysis (patient-level features):")
print("="*80)

corr_features = ['unique_medications', 'ddi_pair_count', 'total_ddi_risk_score', 
                 'ddi_density', 'medication_diversity']
corr_matrix = patient_features[corr_features].corr()

print(corr_matrix)
print("\nNote: High correlation (>0.8) between features may indicate redundancy")
print("="*80)


Correlation analysis (patient-level features):
                      unique_medications  ddi_pair_count  \
unique_medications              1.000000        0.430820   
ddi_pair_count                  0.430820        1.000000   
total_ddi_risk_score            0.442269        0.954786   
ddi_density                     0.120096        0.827837   
medication_diversity            0.986865        0.350902   

                      total_ddi_risk_score  ddi_density  medication_diversity  
unique_medications                0.442269     0.120096              0.986865  
ddi_pair_count                    0.954786     0.827837              0.350902  
total_ddi_risk_score              1.000000     0.686406              0.390119  
ddi_density                       0.686406     1.000000              0.030046  
medication_diversity              0.390119     0.030046              1.000000  

Note: High correlation (>0.8) between features may indicate redundancy


---
## Part 5: Write Features to v3_features

In [41]:
# Write patient-level features to v3_features

patient_features_filename = "patients_features.parquet"
patient_features_uri = f"s3://{DEST_BUCKET}/{V3_FEATURES_DDI_PREFIX}{patient_features_filename}"
logging.info(f"Writing patient features: {patient_features_uri}")

start_time = time.time()

patient_features.to_parquet(
    patient_features_uri,
    engine='pyarrow',
    filesystem=fs,
    compression='snappy',
    index=False
)

elapsed = time.time() - start_time
logging.info(f"Successfully wrote {len(patient_features):,} patient records in {elapsed:.2f}s")

print(f"✓ Patient features written to: {patient_features_uri}")

2025-11-27 12:59:38,629 INFO Writing patient features: s3://med-data/v3_features/ddi/patients_features.parquet
2025-11-27 12:59:38,644 INFO Successfully wrote 10 patient records in 0.01s


✓ Patient features written to: s3://med-data/v3_features/ddi/patients_features.parquet


In [42]:
# Write DDI pair-level features to v3_features

if len(df_ddi_pairs) > 0:
    ddi_pairs_filename = "ddi_pairs_features.parquet"
    ddi_pairs_uri = f"s3://{DEST_BUCKET}/{V3_FEATURES_DDI_PREFIX}{ddi_pairs_filename}"
    logging.info(f"Writing DDI pair features: {ddi_pairs_uri}")
    
    start_time = time.time()
    
    df_ddi_pairs.to_parquet(
        ddi_pairs_uri,
        engine='pyarrow',
        filesystem=fs,
        compression='snappy',
        index=False
    )
    
    elapsed = time.time() - start_time
    logging.info(f"Successfully wrote {len(df_ddi_pairs):,} DDI pair records in {elapsed:.2f}s")
    
    print(f"✓ DDI pair features written to: {ddi_pairs_uri}")
else:
    logging.warning("No DDI pairs to write")
    print("⚠ No DDI pair features to write (no interactions found)")

2025-11-27 12:59:42,013 INFO Writing DDI pair features: s3://med-data/v3_features/ddi/ddi_pairs_features.parquet
2025-11-27 12:59:42,026 INFO Successfully wrote 4 DDI pair records in 0.01s


✓ DDI pair features written to: s3://med-data/v3_features/ddi/ddi_pairs_features.parquet


---
## Part 6: Verification and Summary

In [43]:
# Verify patient features by reading back

logging.info("Verifying patient features...")

start_time = time.time()
df_patient_verify = pd.read_parquet(patient_features_uri, filesystem=fs)
elapsed = time.time() - start_time

assert len(df_patient_verify) == len(patient_features), "Row count mismatch!"
assert len(df_patient_verify.columns) == len(patient_features.columns), "Column count mismatch!"

logging.info(f"✓ Patient features verification successful: {len(df_patient_verify):,} rows in {elapsed:.2f}s")

print("\nPatient Features (first 3 rows):")
print(df_patient_verify.head(3))

2025-11-27 12:59:46,049 INFO Verifying patient features...
2025-11-27 12:59:46,073 INFO ✓ Patient features verification successful: 10 rows in 0.02s



Patient Features (first 3 rows):
   PatientSID  medication_count  unique_medications first_medication_date  \
0        1001                 7                   3   2025-01-01 08:05:00   
1        1002                 3                   3   2025-01-02 07:30:00   
2        1003                 3                   2   2025-01-02 21:10:00   

  last_medication_date  rxout_count  bcma_count  medication_timespan_days  \
0  2025-03-01 10:00:00            3           4                        59   
1  2025-01-22 09:15:00            1           2                        20   
2  2025-02-05 13:25:00            2           1                        33   

   avg_medications_per_day  source_diversity  medication_diversity  \
0                 0.116667                 2              1.448816   
1                 0.142857                 2              1.584963   
2                 0.088235                 2              0.918296   

   ddi_pair_count  ddi_severity_Low  ddi_severity_Moderate  \
0    

In [44]:
# Final feature engineering summary

print("\n" + "="*80)
print("FEATURE ENGINEERING SUMMARY")
print("="*80)

print("\nPATIENT-LEVEL FEATURES:")
print(f"  Output: s3://{DEST_BUCKET}/{V3_FEATURES_DDI_PREFIX}{patient_features_filename}")
print(f"  Patients: {len(patient_features):,}")
print(f"  Features: {len(patient_features.columns)}")
print(f"  Polypharmacy patients: {patient_features['is_polypharmacy'].sum()}")
print(f"  High DDI risk patients: {patient_features['is_high_ddi_risk'].sum()}")
print(f"  Status: ✓ Complete")

if len(df_ddi_pairs) > 0:
    print("\nDDI PAIR-LEVEL FEATURES:")
    print(f"  Output: s3://{DEST_BUCKET}/{V3_FEATURES_DDI_PREFIX}{ddi_pairs_filename}")
    print(f"  DDI pairs: {len(df_ddi_pairs):,}")
    print(f"  Features: {len(df_ddi_pairs.columns)}")
    print(f"  Patients affected: {df_ddi_pairs['PatientSID'].nunique()}")
    print(f"  Temporal overlap: {df_ddi_pairs['temporal_overlap'].sum()} pairs")
    print(f"  Status: ✓ Complete")
else:
    print("\nDDI PAIR-LEVEL FEATURES:")
    print(f"  Status: ⚠ No DDI pairs found (skipped)")

print("\nFEATURE CATEGORIES:")
print("  ✓ Medication profile (count, diversity, burden)")
print("  ✓ Temporal patterns (timespan, frequency)")
print("  ✓ Source system (RxOut, BCMA, diversity)")
print("  ✓ DDI risk (pair count, severity, risk score, density)")
print("  ✓ Clinical indicators (polypharmacy, high risk flags)")
if len(df_ddi_pairs) > 0:
    print("  ✓ Interaction details (type, temporal overlap)")

print("\nUSE CASES SUPPORTED:")
print("  → Patient risk stratification and clustering (05_clustering.ipynb)")
print("  → DDI risk scoring and analysis (06_analysis.ipynb)")
print("  → Future: Predictive modeling, clinical decision support")
print("  → Future: Care coordination analysis (after PhysioNet integration)")

print("\nNEXT STEPS:")
print("  → Run 05_clustering.ipynb to identify patient risk groups")

print("="*80)


FEATURE ENGINEERING SUMMARY

PATIENT-LEVEL FEATURES:
  Output: s3://med-data/v3_features/ddi/patients_features.parquet
  Patients: 10
  Features: 19
  Polypharmacy patients: 0
  High DDI risk patients: 1
  Status: ✓ Complete

DDI PAIR-LEVEL FEATURES:
  Output: s3://med-data/v3_features/ddi/ddi_pairs_features.parquet
  DDI pairs: 4
  Features: 15
  Patients affected: 3
  Temporal overlap: 0 pairs
  Status: ✓ Complete

FEATURE CATEGORIES:
  ✓ Medication profile (count, diversity, burden)
  ✓ Temporal patterns (timespan, frequency)
  ✓ Source system (RxOut, BCMA, diversity)
  ✓ DDI risk (pair count, severity, risk score, density)
  ✓ Clinical indicators (polypharmacy, high risk flags)
  ✓ Interaction details (type, temporal overlap)

USE CASES SUPPORTED:
  → Patient risk stratification and clustering (05_clustering.ipynb)
  → DDI risk scoring and analysis (06_analysis.ipynb)
  → Future: Predictive modeling, clinical decision support
  → Future: Care coordination analysis (after PhysioNet